# Tratando dados dos Filmes

In [34]:
#Instalando Panda

! pip install pandas -q
%pip install scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------  8.1/8.2 MB 77.7 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 29.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ------------------ --------------------- 17.3/36.6 MB 85.6 MB/s eta 0:00:01
   ------------------------------------- -- 34.1/36.6 MB 81.6 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.6 MB 81.3 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.6 MB 81.3 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.6 MB 81.3 MB/s eta 0:00:01
   ---------------------------------------- 36.6/36.6 MB 34.0 MB/s eta 0:00:00

   ---------------------------------------- 0/6 [threadpoolctl]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv('../dados/imdb.csv')
df

,Name,Date,Rate,Votes,Genre,Duration,Type,Certificate,Episodes,Nudity,Violence,Profanity,Alcohol,Frightening
0,No Time to Die,2021,7.6,"107,163","Action, Adventure, Thriller",163.0,Film,PG-13,-,Mild,Moderate,Mild,Mild,Moderate
1,The Guilty,2021,6.3,"64,375","Crime, Drama, Thriller",90.0,Film,R,-,NaN,NaN,Severe,NaN,Moderate
2,The Many Saints of Newark,2021,6.4,"27,145","Crime, Drama",120.0,Film,R,-,Moderate,Severe,Severe,Moderate,Moderate
3,Venom: Let There Be Carnage,2021,6.4,"30,443","Action, Adventure, Sci-Fi",97.0,Film,PG-13,-,NaN,Moderate,Moderate,Mild,Moderate
4,Dune,2021,8.3,"84,636","Action, Adventure, Drama",155.0,Film,PG-13,-,NaN,Moderate,NaN,Mild,Moderate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6173,The Human Centipede II (Full Sequence),2011,3.8,"37,492",Horror,91.0,Film,Not Rated,-,Severe,Severe,Severe,Mild,Severe
6174,Double Indemnity,1944,8.3,"150,448","Crime, Drama, Film-Noir",107.0,Film,Passed,-,NaN,Mild,NaN,Mild,Mild
6175,Before the Devil Knows You're Dead,2007,7.3,"100,668","Crime, Drama, Thriller",117.0,Film,R,-,Severe,Moderate,Severe,Severe,Severe
6176,Queen Bees,2021,6.0,887,"Comedy, Drama, Romance",100.0,Film,PG-13,-,NaN,NaN,Mild,Moderate,NaN


In [36]:
#Identificando as linhas duplicadas
df[df.duplicated(keep=False)]

,Name,Date,Rate,Votes,Genre,Duration,Type,Certificate,Episodes,Nudity,Violence,Profanity,Alcohol,Frightening
3878,Smokey and the Bandit,1977,7.0,"48,070","Action, Adventure, Comedy",96.0,Film,PG,-,Mild,Mild,Moderate,Mild,NaN
3879,Dante's Peak,1997,6.0,"86,689","Action, Adventure, Thriller",108.0,Film,PG-13,-,Mild,Moderate,Mild,Mild,Moderate
3880,"Doogie Howser, M.D.",1997,6.5,"8,443","Comedy, Drama, Family",30.0,Series,TV-PG,97,NaN,Mild,NaN,Mild,Mild
3881,Strange But True,2019,5.9,"4,151",Thriller,96.0,Film,PG-13,-,Mild,Mild,Moderate,Moderate,Moderate
3882,Yes Day,2021,5.7,"20,492","Comedy, Family",86.0,Film,PG,-,NaN,Mild,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6173,The Human Centipede II (Full Sequence),2011,3.8,"37,492",Horror,91.0,Film,Not Rated,-,Severe,Severe,Severe,Mild,Severe
6174,Double Indemnity,1944,8.3,"150,448","Crime, Drama, Film-Noir",107.0,Film,Passed,-,NaN,Mild,NaN,Mild,Mild
6175,Before the Devil Knows You're Dead,2007,7.3,"100,668","Crime, Drama, Thriller",117.0,Film,R,-,Severe,Moderate,Severe,Severe,Severe
6176,Queen Bees,2021,6.0,887,"Comedy, Drama, Romance",100.0,Film,PG-13,-,NaN,NaN,Mild,Moderate,NaN


In [37]:
#tratando as duplicidades
df_limpo_imdb = df.drop_duplicates()

In [38]:
#conferindo se deu certo
df_limpo_imdb.duplicated().sum()

np.int64(0)

In [39]:
#verificando os dados nulos
df_limpo_imdb.isnull().sum()

Name              0
Date              0
Rate              0
Votes             0
Genre             0
Duration        248
Type              0
Certificate     344
Episodes          0
Nudity         1182
Violence        531
Profanity       524
Alcohol         614
Frightening     670
dtype: int64

In [40]:
percentual_nulos = (
    df_limpo_imdb.isnull().mean() * 100
).round(2)

print(percentual_nulos)

Name            0.00
Date            0.00
Rate            0.00
Votes           0.00
Genre           0.00
Duration        4.93
Type            0.00
Certificate     6.84
Episodes        0.00
Nudity         23.51
Violence       10.56
Profanity      10.42
Alcohol        12.21
Frightening    13.33
dtype: float64


In [41]:
# Identificando colunas numéricas e categóricas
colunas_numericas = df_limpo_imdb.select_dtypes(
    include="number"
).columns

colunas_categoricas = df_limpo_imdb.select_dtypes(
    include=["object", "string", "category"]
).columns

# Preenchendo valores numéricos pela mediana
for coluna in colunas_numericas:
    df_limpo_imdb[coluna] = df_limpo_imdb[coluna].fillna(
        df_limpo_imdb[coluna].median()
    )

# Preenchendo categorias pela moda
for coluna in colunas_categoricas:
    moda = df_limpo_imdb[coluna].mode()

    if not moda.empty:
        df_limpo_imdb[coluna] = df_limpo_imdb[coluna].fillna(
            moda.iloc[0]
        )

print("Valores ausentes após o preenchimento:")
print(df_limpo_imdb.isna().sum())

Valores ausentes após o preenchimento:
Name           0
Date           0
Rate           0
Votes          0
Genre          0
Duration       0
Type           0
Certificate    0
Episodes       0
Nudity         0
Violence       0
Profanity      0
Alcohol        0
Frightening    0
dtype: int64


In [42]:
#Removendo colunas que não serão ultilizadas
df_limpo_imdb = df_limpo_imdb.drop(
    columns=[
        'Certificate',
        'Nudity',
        'Violence',
        'Profanity',
        'Alcohol',
        'Frightening'
    ]
)

In [43]:
#conferindo 
df_limpo_imdb.columns

Index(['Name', 'Date', 'Rate', 'Votes', 'Genre', 'Duration', 'Type',
       'Episodes'],
      dtype='str')

In [44]:
# Criando uma coluna ID sequencial
df_limpo_imdb.insert(
    0,
    'ID',
    range(1, len(df_limpo_imdb) + 1)
)

In [45]:
#conferindo
df_limpo_imdb.head(10)

,ID,Name,Date,Rate,Votes,Genre,Duration,Type,Episodes
0,1,No Time to Die,2021,7.6,"107,163","Action, Adventure, Thriller",163.0,Film,-
1,2,The Guilty,2021,6.3,"64,375","Crime, Drama, Thriller",90.0,Film,-
2,3,The Many Saints of Newark,2021,6.4,"27,145","Crime, Drama",120.0,Film,-
3,4,Venom: Let There Be Carnage,2021,6.4,"30,443","Action, Adventure, Sci-Fi",97.0,Film,-
4,5,Dune,2021,8.3,"84,636","Action, Adventure, Drama",155.0,Film,-
5,6,Ted Lasso,2021,8.8,"117,562","Comedy, Drama, Sport",30.0,Series,23
6,7,Free Guy,2021,7.3,"153,835","Action, Adventure, Comedy",115.0,Film,-
7,8,House of the Dragon,2021,No Rate,No Votes,"Action, Adventure, Drama",100.0,Series,10
8,9,What If...?,2021,7.6,"51,959","Animation, Action, Adventure",100.0,Series,18
9,10,Sex Education,2021,8.3,"220,201","Comedy, Drama",45.0,Series,25


In [46]:
#conferindo
print('Total de registros:', len(df_limpo_imdb))
print('Total de IDs únicos:', df_limpo_imdb['ID'].nunique())
print('IDs duplicados:', df_limpo_imdb['ID'].duplicated().sum())
print('Valores nulos no ID:', df_limpo_imdb['ID'].isnull().sum())

Total de registros: 5028
Total de IDs únicos: 5028
IDs duplicados: 0
Valores nulos no ID: 0


In [47]:
# Identificando textos que representam dados ausentes

for coluna in df_limpo_imdb.select_dtypes(include='object').columns:
    valores = df_limpo_imdb[coluna].dropna().unique()

    print(f'\nColuna: {coluna}')
    print([v for v in valores if str(v).strip().lower() in
           ['no rate', 'no votes', 'none', 'null', 'n/a', 'na']])


Coluna: Name
[]

Coluna: Rate
['No Rate']

Coluna: Votes
['No Votes']

Coluna: Genre
[]

Coluna: Type
[]

Coluna: Episodes
[]


C:\Users\alexo\AppData\Local\Temp\ipykernel_11912\3071286132.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for coluna in df_limpo_imdb.select_dtypes(include='object').columns:


In [48]:
import numpy as np

# Substituindo textos que representam valores ausentes
df_limpo_imdb = df_limpo_imdb.replace({
    'Rate': {'No Rate': np.nan},
    'Votes': {'No Votes': np.nan}
})

In [49]:
#convertendo as colunas para valores numericos
df_limpo_imdb['Rate'] = pd.to_numeric(
    df_limpo_imdb['Rate'],
    errors='coerce'
)

df_limpo_imdb['Votes'] = pd.to_numeric(
    df_limpo_imdb['Votes'],
    errors='coerce'
)

In [50]:
#conferindo o resultado
print('Valores ausentes após a conversão:')

display(
    df_limpo_imdb[['Rate', 'Votes']].isnull().sum()
)

print('\nTipos de dados:')

display(
    df_limpo_imdb[['Rate', 'Votes']].dtypes
)

Valores ausentes após a conversão:


Rate      153
Votes    4816
dtype: int64


Tipos de dados:


Rate     float64
Votes    float64
dtype: object

# Resolvendo o problema da virgula em votes

In [51]:
# Recuperando os valores originais de Votes
df_limpo_imdb['Votes'] = (
    df.loc[df_limpo_imdb.index, 'Votes']
    .astype('string')
    .str.replace(',', '', regex=False)
)

# Convertendo para numérico
df_limpo_imdb['Votes'] = pd.to_numeric(
    df_limpo_imdb['Votes'],
    errors='coerce'
).astype('Int64')

In [52]:
df_limpo_imdb['Rate'].isnull().sum()

np.int64(153)

In [53]:
#Conferindo novamente
print('Valores ausentes:')

display(df_limpo_imdb[['Rate', 'Votes']].isnull().sum())

print('\nTipos de dados:')

display(df_limpo_imdb[['Rate', 'Votes']].dtypes)

print('\nEstatísticas das colunas:')

display(df_limpo_imdb[['Rate', 'Votes']].describe())

Valores ausentes:


Rate     153
Votes    153
dtype: int64


Tipos de dados:


Rate     float64
Votes      Int64
dtype: object


Estatísticas das colunas:


,Rate,Votes
count,4875.000000,4875.0
mean,6.929764,136903.187487
std,1.070860,208745.150026
min,1.200000,31.0
25%,6.300000,16054.5
50%,7.000000,64857.0
75%,7.700000,172466.0
max,9.700000,2474122.0


In [54]:
# Verificando registros sem Rate e Votes
df_limpo_imdb[
    df_limpo_imdb['Rate'].isnull() &
    df_limpo_imdb['Votes'].isnull()
].shape

(153, 9)

In [55]:
#identificando colunas de texto
colunas_categoricas = df_limpo_imdb.select_dtypes(
    include=['object', 'str', 'category']
).columns

print(colunas_categoricas)

Index(['Name', 'Genre', 'Type', 'Episodes'], dtype='str')


In [56]:
df_limpo_imdb['Episodes'].value_counts().head(10)

Episodes
-     3586
10      67
20      54
16      44
8       39
12      39
18      35
6       33
30      33
9       30
Name: count, dtype: int64

In [57]:
# Visualizar os 10 primeiros gêneros
df['Genre'].head(10)

0     Action, Adventure, Thriller
1          Crime, Drama, Thriller
2                    Crime, Drama
3       Action, Adventure, Sci-Fi
4        Action, Adventure, Drama
5            Comedy, Drama, Sport
6       Action, Adventure, Comedy
7        Action, Adventure, Drama
8    Animation, Action, Adventure
9                   Comedy, Drama
Name: Genre, dtype: str

In [58]:
# Verificar quantos gêneros diferentes existem
df['Genre'].nunique()

377

In [59]:
# Visualizar os gêneros mais frequentes
df['Genre'].value_counts().head(15)

Genre
Comedy                          268
Drama                           259
Crime, Drama, Mystery           220
Comedy, Drama                   199
Drama, Romance                  189
Action, Crime, Drama            171
Comedy, Drama, Romance          156
Animation, Adventure, Comedy    154
Comedy, Romance                 148
Action, Adventure, Sci-Fi       144
Action, Adventure, Drama        140
Crime, Drama, Thriller          120
Action, Adventure, Comedy       114
Animation, Action, Adventure    108
Horror, Thriller                107
Name: count, dtype: int64

In [60]:
# Separar os gêneros individuais
generos = df_limpo_imdb['Genre'].dropna().str.split(',').explode().str.strip()

# Contar a frequência de cada gênero
generos.value_counts()

Genre
Drama          2655
Comedy         1587
Action         1392
Adventure      1087
Crime           982
Thriller        821
Horror          719
Mystery         634
Romance         592
Fantasy         488
Sci-Fi          485
Animation       323
Family          279
Biography       240
History         151
Music            86
Sport            65
War              57
Western          52
Musical          42
Reality-TV       33
Short            24
Documentary      23
Game-Show        14
Film-Noir         5
Talk-Show         2
News              1
Name: count, dtype: int64

In [63]:
generos_lista = (
    df_limpo_imdb['Genre']
    .fillna('')
    .str.split(',')
    .apply(lambda lista: [g.strip() for g in lista if g.strip()])
)

In [64]:
mlb = MultiLabelBinarizer()

generos_transformados = mlb.fit_transform(generos_lista)

In [65]:
for i, genero in enumerate(mlb.classes_):
    df_limpo_imdb[genero] = generos_transformados[:, i]

In [66]:
#Remove a coluna original genero
df_limpo_imdb.drop(columns=['Genre'], inplace=True)

C:\Users\alexo\AppData\Local\Temp\ipykernel_11912\343332826.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_limpo_imdb.drop(columns=['Genre'], inplace=True)


In [67]:
df_limpo_imdb.head()

,ID,Name,Date,Rate,Votes,Duration,Type,Episodes,Action,Adventure,...,News,Reality-TV,Romance,Sci-Fi,Short,Sport,Talk-Show,Thriller,War,Western
0,1,No Time to Die,2021,7.6,107163,163.0,Film,-,1,1,...,0,0,0,0,0,0,0,1,0,0
1,2,The Guilty,2021,6.3,64375,90.0,Film,-,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,The Many Saints of Newark,2021,6.4,27145,120.0,Film,-,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,Venom: Let There Be Carnage,2021,6.4,30443,97.0,Film,-,1,1,...,0,0,0,1,0,0,0,0,0,0
4,5,Dune,2021,8.3,84636,155.0,Film,-,1,1,...,0,0,0,0,0,0,0,0,0,0


# encoding na coluna type

In [68]:
df_limpo_imdb['Type'].value_counts()

Type
Film      3586
Series    1442
Name: count, dtype: int64

In [69]:
# Aplicar One-Hot Encoding na coluna Type
df_limpo_imdb = pd.get_dummies(
    df_limpo_imdb,
    columns=['Type'],
    dtype=int
)

In [70]:
df_limpo_imdb[['Type_Film', 'Type_Series']].head(10)

,Type_Film,Type_Series
0,1,0
1,1,0
2,1,0
3,1,0
4,1,0
5,0,1
6,1,0
7,0,1
8,0,1
9,0,1


# tratamento de encoding na coluna episode

In [71]:
df_limpo_imdb['Episodes'].value_counts().head(15)

Episodes
-     3586
10      67
20      54
16      44
8       39
12      39
18      35
6       33
30      33
9       30
11      26
24      23
26      21
25      20
13      19
Name: count, dtype: int64

In [73]:
df_limpo_imdb['Episodes'] = df_limpo_imdb['Episodes'].replace('-', pd.NA)

In [74]:
df_limpo_imdb['Episodes'] = pd.to_numeric(
    df_limpo_imdb['Episodes']
).astype('Int64')

In [76]:
df_limpo_imdb['Episodes'].isna().sum()

np.int64(3586)

In [77]:
# Verificar o tipo de dado
df_limpo_imdb['Date'].dtype

dtype('int64')

In [78]:
df_limpo_imdb['Duration'].dtype

dtype('float64')

In [80]:
df_limpo_imdb.dtypes

ID               int64
Name               str
Date             int64
Rate           float64
Votes            Int64
Duration       float64
Episodes         Int64
Action           int64
Adventure        int64
Animation        int64
Biography        int64
Comedy           int64
Crime            int64
Documentary      int64
Drama            int64
Family           int64
Fantasy          int64
Film-Noir        int64
Game-Show        int64
History          int64
Horror           int64
Music            int64
Musical          int64
Mystery          int64
News             int64
Reality-TV       int64
Romance          int64
Sci-Fi           int64
Short            int64
Sport            int64
Talk-Show        int64
Thriller         int64
War              int64
Western          int64
Type_Film        int64
Type_Series      int64
dtype: object

In [81]:
df_limpo_imdb.head()

,ID,Name,Date,Rate,Votes,Duration,Episodes,Action,Adventure,Animation,...,Romance,Sci-Fi,Short,Sport,Talk-Show,Thriller,War,Western,Type_Film,Type_Series
0,1,No Time to Die,2021,7.6,107163,163.0,<NA>,1,1,0,...,0,0,0,0,0,1,0,0,1,0
1,2,The Guilty,2021,6.3,64375,90.0,<NA>,0,0,0,...,0,0,0,0,0,1,0,0,1,0
2,3,The Many Saints of Newark,2021,6.4,27145,120.0,<NA>,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,4,Venom: Let There Be Carnage,2021,6.4,30443,97.0,<NA>,1,1,0,...,0,1,0,0,0,0,0,0,1,0
4,5,Dune,2021,8.3,84636,155.0,<NA>,1,1,0,...,0,0,0,0,0,0,0,0,1,0


In [82]:
df_limpo_imdb.to_csv('imdb_tratado.csv', index=False, encoding='utf-8-sig')